In [ ]:
# STEP 0 — Install & Import Libraries

In [ ]:
# Install dependencies (Google Colab users uncomment)
# !pip install nltk gensim matplotlib sklearn tensorflow seaborn

import nltk
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.corpus import treebank
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, TimeDistributed, SimpleRNN, LSTM, GRU, Bidirectional
from sklearn.model_selection import train_test_split


In [ ]:
#STEP 1 — Download & Load Dataset (POS TAGGING)

In [ ]:
nltk.download("treebank")
nltk.download("universal_tagset")

tagged_sentences = treebank.tagged_sents(tagset="universal")

X, Y = [], []

for sentence in tagged_sentences:
    words = [token for token, tag in sentence]
    tags  = [tag for token, tag in sentence]
    X.append(words)
    Y.append(tags)

print("Total sentences:", len(X))
print("Sample sentence:", X[0])
print("Sample tags     :", Y[0])


[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Unzipping corpora/treebank.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


Total sentences: 3914
Sample sentence: ['Pierre', 'Vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'Nov.', '29', '.']
Sample tags     : ['NOUN', 'NOUN', '.', 'NUM', 'NOUN', 'ADJ', '.', 'VERB', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'ADJ', 'NOUN', 'NOUN', 'NUM', '.']


In [ ]:
#STEP 2 — Convert Text → Integer Encoding

In [ ]:
word_tokenizer = Tokenizer()
word_tokenizer.fit_on_texts(X)
X_encoded = word_tokenizer.texts_to_sequences(X)

tag_tokenizer = Tokenizer()
tag_tokenizer.fit_on_texts(Y)
Y_encoded = tag_tokenizer.texts_to_sequences(Y)

VOCAB_SIZE = len(word_tokenizer.word_index) + 1
NUM_CLASSES = len(tag_tokenizer.word_index) + 1


In [ ]:
#STEP 3 — Padding Sequences

In [ ]:
MAX_SEQ_LENGTH = 80  # Hyperparameter

X_padded = pad_sequences(X_encoded, maxlen=MAX_SEQ_LENGTH, padding="post")
Y_padded = pad_sequences(Y_encoded, maxlen=MAX_SEQ_LENGTH, padding="post")
Y_onehot = to_categorical(Y_padded, num_classes=NUM_CLASSES)


In [ ]:
#STEP 4 — Load Pretrained Word Embeddings (GloVe 100d)

In [ ]:
# Download GloVe embedding (automatically downloads)
import os, zipfile, requests

if not os.path.exists("glove.6B.100d.txt"):
    url = "http://nlp.stanford.edu/data/glove.6B.zip"
    with open("glove.zip", "wb") as f:
        f.write(requests.get(url).content)
    with zipfile.ZipFile("glove.zip", "r") as zip_ref:
        zip_ref.extractall(".")

EMBEDDING_DIM = 100
embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM))

glove = {}
with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        glove[values[0]] = np.asarray(values[1:], dtype="float32")

for word, index in word_tokenizer.word_index.items():
    vector = glove.get(word.lower())
    if vector is not None:
        embedding_matrix[index] = vector


In [ ]:
#STEP 5 — Models (Vanilla RNN, LSTM, GRU, BiLSTM)

In [ ]:
#Build Model Function (Reusable)

In [ ]:
def build_model(recurrent_layer):
    model = Sequential()

    model.add(Embedding(VOCAB_SIZE, EMBEDDING_DIM,
                        weights=[embedding_matrix],
                        input_length=MAX_SEQ_LENGTH,
                        trainable=True))

    model.add(recurrent_layer)
    model.add(TimeDistributed(Dense(NUM_CLASSES, activation="softmax")))

    model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model


In [ ]:
#Train / Validate / Test Split
X_train, X_test, Y_train, Y_test = train_test_split(X_padded, Y_onehot, test_size=0.10)
X_train, X_val,  Y_train, Y_val  = train_test_split(X_train, Y_train, test_size=0.10)


In [ ]:
#Train Models
models = {
    "Vanilla RNN": SimpleRNN(64, return_sequences=True),
    "LSTM": LSTM(64, return_sequences=True),
    "GRU": GRU(64, return_sequences=True),
    "Bidirectional LSTM": Bidirectional(LSTM(64, return_sequences=True))
}

trained_models = {}

for name, layer in models.items():
    print(f"\n Training {name}...\n")
    model = build_model(layer)
    model.fit(X_train, Y_train, epochs=2, batch_size=128, validation_data=(X_val, Y_val))
    trained_models[name] = model



🚀 Training Vanilla RNN...

Epoch 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 259ms/step - accuracy: 0.6585 - loss: 1.2980 - val_accuracy: 0.8485 - val_loss: 0.5288
Epoch 2/2
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 122ms/step - accuracy: 0.8644 - loss: 0.4808 - val_accuracy: 0.9024 - val_loss: 0.3663

🚀 Training LSTM...

Epoch 1/2
25/25 ━━━━━━━━━━━━━━━━━━━━ 16s 429ms/step - accuracy: 0.6414 - loss: 1.3637 - val_accuracy: 0.7705 - val_loss: 0.6845
Epoch 2/2
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 224ms/step - accuracy: 0.7904 - loss: 0.6498 - val_accuracy: 0.8400 - val_loss: 0.5530

🚀 Training GRU...

Epoch 1/2
25/25 ━━━━━━━━━━━━━━━━━━━━ 17s 395ms/step - accuracy: 0.6757 - loss: 1.3705 - val_accuracy: 0.8296 - val_loss: 0.5865
Epoch 2/2
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 243ms/step - accuracy: 0.8565 - loss: 0.5331 - val_accuracy: 0.9051 - val_loss: 0.3739

🚀 Training Bidirectional LSTM...

Epoch 1/2
25/25 ━━━━━━━━━━━━━━━━━━━━ 25s 628ms/step - accuracy: 0.6537 - loss: 1.3309 - val_accuracy: 0.7913 - val_loss: 0.6338
Epoch 2/2
25/25 ━━━━━━━━━━━━━━━━━━━━ 18s 

In [ ]:
#STEP 6 — Model Evaluation
for name, model in trained_models.items():
    loss, acc = model.evaluate(X_test, Y_test)
    print(f"\n {name} — Test Accuracy: {acc:.4f}")


13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9040 - loss: 0.3565

 Vanilla RNN — Test Accuracy: 0.9057
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8458 - loss: 0.5374

 LSTM — Test Accuracy: 0.8485
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9076 - loss: 0.3661

 GRU — Test Accuracy: 0.9083
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.8784 - loss: 0.4436

 Bidirectional LSTM — Test Accuracy: 0.8807


In [ ]:
#Model                      #Learns context?                      #Handles long sequences?                 #Bidirectional?
#Vanilla RNN                     X weak memory                          X vanishing gradient                    No
#LSTM                              Yes                                     Yes                                   No
#GRU                               Yes, faster than LSTM                 Yes                                    No
#BiLSTM                            Yes, best contextual understanding        Yes                                Yes